## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [ ]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [ ]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df


# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean


# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kabkot"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired


# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    


# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]


# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


# Mendapatkan nama komponen sesuai dengan template evaluasi
KOMPONEN_MAPPING = {
    "adhb_pdrb": "PDRB ADHB",
    "adhk_pdrb": "PDRB ADHK",
    "q-to-q_pdrb": "PDRB ADHK",
    "y-on-y_pdrb": "PDRB ADHK",
    "c-to-c_pdrb": "PDRB ADHK",
    "pkrt": "PKRT",
    "implisit_q-to-q_pkrt": "PKRT",
    "implisit_y-on-y_pkrt": "PKRT",
    "pklnprt": "PKLNPRT",
    "implisit_q-to-q_pklnprt": "PKLNPRT",
    "implisit_y-on-y_pklnprt": "PKLNPRT",
    "pkp": "PKP",
    "implisit_q-to-q_pkp": "PKP",
    "implisit_y-on-y_pkp": "PKP",
    "pmtb": "PMTB",
    "implisit_q-to-q_pmtb": "PMTB",
    "implisit_y-on-y_pmtb": "PMTB",
    "distribusi_ekspor": "Ekspor",
    "q-to-q_ekspor": "Ekspor",
    "y-on-y_ekspor": "Ekspor",
    "c-to-c_ekspor": "Ekspor",
    "implisit_y-on-y_ekspor": "Ekspor",
    "distribusi_impor": "Impor",
    "q-to-q_impor": "Impor",
    "y-on-y_impor": "Impor",
    "c-to-c_impor": "Impor",
    "implisit_y-on-y_impor": "Impor",
    "pi": "PI",
    "implisit_q-to-q_pi": "PI",
    "implisit_y-on-y_pi": "PI",
    "net_ekspor": "NET Ekspor",
    "implisit_q-to-q_net_ekspor": "NET Ekspor",
    "implisit_y-on-y_net_ekspor": "NET Ekspor"
}

def get_komponen(text):
    text_lower = text.lower()
    
    for key, value in KOMPONEN_MAPPING.items():
        if key in text_lower:
            return value
    
    return None


# Mendapatkan nama periode sesuai dengan template evaluasi
def get_periode(text):
    text_lower = text.lower()
    
    # Cari pola qX-YY
    match_quarter = re.search(r'q([1-4])-(\d{2})', text_lower)
    if match_quarter:
        quarter = match_quarter.group(1)
        year = match_quarter.group(2)
        full_year = "20" + year
        return f"Q{quarter} {full_year}"
    
    # Cari tahun 4 digit
    match_year = re.search(r'(20\d{2})', text_lower)
    if match_year:
        return match_year.group(1)
    
    return None


# Mendapatkan nama indikator sesuai dengan template evaluasi
INDIKATOR_MAPPING = {
    "adhb": "Nilai",
    "adhk": "Nilai",
    "distribusi": "Distribusi",
    "implisit_q-to-q": "Implisit Q-to-Q",
    "implisit_y-on-y": "Implisit Y-on-Y",
    "sog_q": "SoG Q-to-Q",
    "sog_y-on-y": "SoG Y-on-Y",
    "q-to-q": "Growth Q-to-Q",
    "y-on-y": "Growth Y-on-Y",
    "c-to-c": "Growth C-to-C",    
}

def get_indikator(text):
    text_lower = text.lower()
    
    for key, value in INDIKATOR_MAPPING.items():
        if key in text_lower:
            return value
    
    return None

#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [ ]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [ ]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Diskrepansi Prov vs KabKot")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [ ]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2026
            disk_pdrb_tresh = 3
            if any(q in col for q in ["q1", "q2"]):
                if pd.notna(val) and (val < -disk_pdrb_tresh or val > disk_pdrb_tresh):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = f"Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-{disk_pdrb_tresh}%)"
                    elif "adhk" in col:
                        evaluasi_text = f"Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-{disk_pdrb_tresh}%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "komponen": get_komponen(col), 
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col),
                        "nilai": val,
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # # ADHB dan ADHK Q3 2025
            # elif "q3" in col:
            #     if pd.notna(val) and (val < -3 or val > 3):
            #         # Pisahkan nama kolom jadi komponen
            #         # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
            #         parts = col.split("_")
                    
            #         periode = parts[-1]  # ambil q1 / q2

            #         # tentukan evaluasi berdasarkan isi col
            #         if "adhb" in col:
            #             evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
            #         elif "adhk" in col:
            #             evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

            #         record = {
            #             "kode": df_clean.at[idx, "kode"],
            #             "provinsi": df_clean.at[idx, "provinsi"],
            #             "periode": periode,
            #             "nilai": val,
            #             "kolom": col,  # tambahan biar tau sumbernya
            #             "evaluasi": evaluasi_text
            #         }
            #         evaluasi_diskrepansi.append(record)
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1 dan Q2 2026
            disk_komp_tresh = 5
            if pd.notna(val) and (val < -disk_komp_tresh or val > disk_komp_tresh):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = f"Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-{disk_komp_tresh}%)"
                elif "adhk" in col:
                    evaluasi_text = f"Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-{disk_komp_tresh}%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "komponen": get_komponen(col),
                    "periode": get_periode(periode),
                    "indikator": get_indikator(col),
                    "nilai": val,
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

#### 3. Evaluasi distribusi

In [ ]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = re.sub(r'_[pk]$', '', col)
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

dist_tresh = 5

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_p")][0]
        col1 = [c for c in cols if c.endswith("_k")][0]
        
        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1 dan Q2 2026
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    }
                    evaluasi_dist.append(record)
                elif selisih > dist_tresh:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": f"Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-{dist_tresh} point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [ ]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = re.sub(r'_[pk]$', '', col)
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_p")][0]
        col1 = [c for c in cols if c.endswith("_k")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    qtoq_pdrb_tresh = 1
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > qtoq_pdrb_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{qtoq_pdrb_tresh} point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    qtoq_komp_tresh = 1
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > qtoq_komp_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{qtoq_komp_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    yony_pdrb_tresh = 0.01
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2026
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > yony_pdrb_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": f"Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{yony_pdrb_tresh} point)"
                            }
                            evaluasi_growth.append(record)
                    # # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.01:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    yony_komp_tresh = 0.5
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2026
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > yony_komp_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": f"Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{yony_komp_tresh} point)"
                            }
                            evaluasi_growth.append(record)
                    # # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.5:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                ctoc_pdrb_tresh = 1
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > ctoc_pdrb_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-{ctoc_pdrb_tresh} point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    ctoc_komp_tresh = 1
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > ctoc_komp_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-{ctoc_komp_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                qtoq_impl_tresh = 1
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > qtoq_impl_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{qtoq_impl_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                yony_impl_tresh = 1
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > yony_impl_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{yony_impl_tresh} point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "provinsi")

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [ ]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Revisi & Growth")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

#### 2. Evaluasi revisi ADHB dan ADHK

In [ ]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

revisi_harga_tresh_pdrb = 0.02
revisi_harga_tresh_komponen = 0.05

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = round(abs((v1 - v0) / v0), 14)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                # evaluasi untuk PDRB
                elif "PDRB" in get_komponen(col0) and selisih > revisi_harga_tresh_pdrb:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": f"{selisih} > {revisi_harga_tresh_pdrb} !! Nilai revisi lebih dari (+-{revisi_harga_tresh_pdrb * 100}%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)
                # evaluasi untuk komponen
                elif "PDRB" not in get_komponen(col0) and selisih > revisi_harga_tresh_komponen:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": f"{selisih} > {revisi_harga_tresh_komponen} !! Nilai revisi lebih dari (+-{revisi_harga_tresh_komponen * 100}%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

#### 3. Evaluasi revisi distribusi

In [ ]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

revisi_dist_tresh = 5

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif (selisih > 5 and 'pi' not in col0 and 'net_ekspor' not in col0):
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "komponen": get_komponen(col0),
                        "periode": get_periode(periode),
                        "indikator": get_indikator(col0),
                        "nilai": v1,
                        "evaluasi": f"Nilai revisi lebih dari (+-{revisi_dist_tresh} point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

unpaired_columns = get_unpaired_columns(revisi_dist_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # Ekspor dan Impor
        if "net_ekspor" not in col and ("ekspor" in col or "impor" in col):
            dist_xm_tresh_min = 10
            dist_xm_tresh_max = 80
            if pd.notna(val) and (val < dist_xm_tresh_min or val > dist_xm_tresh_max):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "komponen": get_komponen(col),
                    "periode": get_periode(col.split("_")[-1]),
                    "indikator": get_indikator(col),
                    "nilai": val,
                    "evaluasi": f"Distribusi lebih dari {dist_xm_tresh_max} persen atau kurang dari {dist_xm_tresh_min} persen?"
                }
                evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                revisi_growth_pdrb_tresh_yony = 0.03
                revisi_growth_pdrb_tresh_qtoq = 3
                revisi_growth_pdrb_tresh_ctoc = 3
                revisi_growth_pdrb_tresh_implisit_yony = 3
                revisi_growth_pdrb_tresh_implisit_qtoq = 3
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    # if type(v0) == str or type(v1) == str:
                    #     print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                    #     print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = round(abs(v1 - v0), 14)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Y-on-Y" in get_indikator(col0) and "Implisit" not in get_indikator(col0) and selisih > revisi_growth_pdrb_tresh_yony:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_pdrb_tresh_yony} !! Nilai revisi lebih dari (+-{revisi_growth_pdrb_tresh_yony} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Q-to-Q" in get_indikator(col0) and "Implisit" not in get_indikator(col0) and selisih > revisi_growth_pdrb_tresh_qtoq:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_pdrb_tresh_qtoq} !! Nilai revisi lebih dari (+-{revisi_growth_pdrb_tresh_qtoq} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "C-to-C" in get_indikator(col0) and selisih > revisi_growth_pdrb_tresh_ctoc:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_pdrb_tresh_ctoc} !! Nilai revisi lebih dari (+-{revisi_growth_pdrb_tresh_ctoc} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    # Implisit
                    elif "Implisit Y-on-Y" in get_indikator(col0) and selisih > revisi_growth_pdrb_tresh_implisit_yony:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_pdrb_tresh_implisit_yony} !! Nilai revisi lebih dari (+-{revisi_growth_pdrb_tresh_implisit_yony} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Implisit Q-to-Q" in get_indikator(col0) and selisih > revisi_growth_pdrb_tresh_implisit_qtoq:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_pdrb_tresh_implisit_qtoq} !! Nilai revisi lebih dari (+-{revisi_growth_pdrb_tresh_implisit_qtoq} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                revisi_growth_komp_tresh_yony = 1
                revisi_growth_komp_tresh_qtoq = 4
                revisi_growth_komp_tresh_ctoc = 4
                revisi_growth_komp_tresh_implisit_yony = 4
                revisi_growth_komp_tresh_implisit_qtoq = 4
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    # if type(v0) == str or type(v1) == str:
                    #     print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                    #     print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = round(abs(v1 - v0), 14)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Y-on-Y" in get_indikator(col0) and "Implisit" not in get_indikator(col0) and selisih > revisi_growth_komp_tresh_yony:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_komp_tresh_yony} !! Nilai revisi lebih dari (+-{revisi_growth_komp_tresh_yony} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Q-to-Q" in get_indikator(col0) and "Implisit" not in get_indikator(col0) and selisih > revisi_growth_komp_tresh_qtoq:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_komp_tresh_qtoq} !! Nilai revisi lebih dari (+-{revisi_growth_komp_tresh_qtoq} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "C-to-C" in get_indikator(col0) and selisih > revisi_growth_komp_tresh_ctoc:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_komp_tresh_ctoc} !! Nilai revisi lebih dari (+-{revisi_growth_komp_tresh_ctoc} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    # Implisit
                    elif "Implisit Y-on-Y" in get_indikator(col0) and selisih > revisi_growth_komp_tresh_implisit_yony:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_komp_tresh_implisit_yony} !! Nilai revisi lebih dari (+-{revisi_growth_komp_tresh_implisit_yony} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif "Implisit Q-to-Q" in get_indikator(col0) and selisih > revisi_growth_komp_tresh_implisit_qtoq:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"{selisih} > {revisi_growth_komp_tresh_implisit_qtoq} !! Nilai revisi lebih dari (+-{revisi_growth_komp_tresh_implisit_qtoq} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                if pd.notna(v0) and pd.notna(v1):
                    # if type(v0) == str or type(v1) == str:
                    #     print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                    #     print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        revisi_sog_pi_tresh = 2
                        if v1 < -revisi_sog_pi_tresh or v1 > revisi_sog_pi_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": f"SoG PI lebih dari +-{revisi_sog_pi_tresh} persen"
                            }
                            evaluasi_revisi_growth.append(record)
                    elif 'net_ekspor' in col0:
                        revisi_sog_net_ekspor_tresh = 10
                        if v1 < -revisi_sog_net_ekspor_tresh or v1 > revisi_sog_net_ekspor_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "komponen": get_komponen(col0),
                                "periode": get_periode(periode),
                                "indikator": get_indikator(col0),
                                "nilai": v1,
                                "evaluasi": f"SoG PI lebih dari +-{revisi_sog_net_ekspor_tresh} persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kabkot" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kabkot dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kabkot")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                growth_tresh = 4
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > growth_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "komponen": get_komponen(col0),
                            "periode": get_periode(periode),
                            "indikator": get_indikator(col0),
                            "nilai": v1,
                            "evaluasi": f"Selisih dengan nilai provinsi cukup jauh (+-{growth_tresh} point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            growth_pi_tresh = 2
            if pd.notna(val) and (val < -growth_pi_tresh or val > growth_pi_tresh):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "komponen": get_komponen(col),
                    "periode": get_periode(col.split("_")[-1]),
                    "indikator": get_indikator(col),
                    "nilai": val,
                    "evaluasi": f"SoG PI lebih dari +-{growth_pi_tresh} persen"
                }
                evaluasi_value_growth.append(record)
        # Net Ekspor
        elif "net_ekspor" in col:
            growth_net_ekspor_tresh = 10
            if pd.notna(val) and (val < -growth_net_ekspor_tresh or val > growth_net_ekspor_tresh):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "komponen": get_komponen(col),
                    "periode": get_periode(col.split("_")[-1]),
                    "indikator": get_indikator(col),
                    "nilai": val,
                    "evaluasi": f"SoG PI lebih dari +-{growth_net_ekspor_tresh} persen"
                }
                evaluasi_value_growth.append(record)
        # Ekspor dan Impor
        elif "net_ekspor" not in col and ("ekspor" in col or "impor" in col):
            growth_xm_tresh = 25
            if pd.notna(val) and (val < -growth_xm_tresh or val > growth_xm_tresh):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "komponen": get_komponen(col),
                    "periode": get_periode(col.split("_")[-1]),
                    "indikator": get_indikator(col),
                    "nilai": val,
                    "evaluasi": f"Pertumbuhan lebih dari +-{growth_xm_tresh} persen?"
                }
                evaluasi_value_growth.append(record)


# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "kabupaten/kota")